# Papaloizou–Pringle disk — theory

Set the parameters in the next cell, run all. Physics comes from `disk_model.py`.

In [ ]:
# ============================ PARAMETERS ============================
USE_DISK_MODEL_DEFAULTS = True   # True:  mirror disk_model.py - its DiskModel field
                                 #        defaults plus RUN_SETUP, i.e. exactly what
                                 #        the athinput was generated from
                                 #        i.e. exactly what the athinput was built from
                                 # False: use the PARAMS and GRID blocks below

PARAMS = dict(
    M_bh     = 4.5e7,   # black hole mass          [M_sun]
    T_0      = 5.0e4,   # reference temperature    [K]   (sets the UNITS, not the gas)
    mu       = 0.6,     # mean molecular weight    (0.6 ionised / 2.3 molecular)
    chi      = 5.0e2,   # r_0 / r_g
    rho_0    = 1.0e-13, # reference density        [g/cm^3]
    gamma    = 1.3,     # adiabatic index          -.
    C_prime  = 0.2,     # disk thickness (<0.5)   -'  H/r = sqrt((gamma-1)(0.5-C'))
    r_center = 1.0,     # density maximum          [r_0]
    alpha    = 1.0e-3,  # Shakura-Sunyaev viscosity
    rho_atm  = 1.0e-4,  # ambient density
)
GRID = dict(nx1=176, nx2=128)    # resolution used for the grid/force diagnostics
# ====================================================================

import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import disk_model
from disk_model import DiskModel, orbits_to_accrete, PC, YR, K_B, M_P

try:
    from ipywidgets import interact, FloatSlider, FloatLogSlider, IntSlider
    HAVE_WIDGETS = True
except ImportError:                                   # notebook still runs without them
    HAVE_WIDGETS = False
    def interact(f, **kw):
        return f(**{k: getattr(v, "value", v) for k, v in kw.items()})
    FloatSlider = FloatLogSlider = IntSlider = lambda **kw: kw.get("value")

# Categorical hues, fixed order, validated for protan/deutan separation and contrast.
INK, MUTED, SURFACE, GUIDE = "#1a1a19", "#6b6b68", "#fcfcfb", "#b8b8b4"
CAT = ["#3D9BC7", "#D55E00", "#7A4FBF", "#B0003A"]
mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "600",
    "axes.labelcolor": INK, "text.color": INK,
    "axes.edgecolor": "#d8d8d4", "axes.linewidth": 0.8,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.grid": True, "grid.color": "#e8e8e4", "grid.linewidth": 0.6,
    "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "legend.fontsize": 9,
    "lines.linewidth": 2.0, "figure.dpi": 120,
})

def build(**over):
    """A DiskModel from the block above, with per-call overrides."""
    kw = {} if USE_DISK_MODEL_DEFAULTS else dict(PARAMS)
    kw.update(over)
    return DiskModel(**kw)

if USE_DISK_MODEL_DEFAULTS:                # keep the grid in step with the athinput too
    GRID = {k: disk_model.RUN_SETUP[k] for k in ("nx1", "nx2")}

def band(ax, m, g=None):
    ax.axvspan(m.r_in, m.r_out, color="#000000", alpha=0.035, lw=0, zorder=0)
    ax.axvline(m.r_center, color=GUIDE, ls=":", lw=1.2, zorder=1)
    if g:
        for x in (g["x1min"], g["x1max"]):
            ax.axvline(x, color=GUIDE, ls="--", lw=1.0, zorder=1)

def cells(m, g, nx1):
    """The volume-centroid radii Athena++ actually uses (x1v), on this grid."""
    rf = np.linspace(g["x1min"], g["x1max"], nx1 + 1)
    return rf, (2/3) * (rf[1:]**3 - rf[:-1]**3) / (rf[1:]**2 - rf[:-1]**2)

m = build()
g = m.grid(**GRID)
print("defaults from disk_model.py" if USE_DISK_MODEL_DEFAULTS else "PARAMS block")

## Summary

In [ ]:
def summarise(m, g):
    hr = np.sqrt((m.gamma - 1) * (0.5 - m.C_prime))
    cs_mid = np.sqrt(m.gamma * m.p_disk(m.r_center) / m.rho_disk(m.r_center)) * m.cs0
    T_mid = m.mu * M_P * cs_mid**2 / (m.gamma * K_B)
    rf, rv = cells(m, g, g["nx1"])
    vol = 0.5 * (rf[1:]**2 - rf[:-1]**2) * 2*np.pi
    M = (m.rho_disk(rv) * vol).sum() * m.mass_scale
    N = orbits_to_accrete(m.alpha, m.gamma, m.C_prime)
    rows = [
        ("cs0  (velocity unit)", f"{m.cs0:.4e} cm/s"),
        ("L_0  (length unit)",   f"{m.L_0:.4e} cm = {m.L_0/PC:.5f} pc"),
        ("t_0  (time unit)",     f"{m.T_scale:.4e} s = {m.T_scale/YR:.1f} yr"),
        ("beta",                 f"{m.beta:.4e}"),
        ("n_poly",               f"{m.n_poly:.3f}"),
        ("disk r_in .. r_out",  f"{m.r_in:.4f} .. {m.r_out:.4f}  "
                                 f"({m.r_in*m.L_0/PC:.5f} .. {m.r_out*m.L_0/PC:.5f} pc)"),
        ("domain x1min..x1max",  f"{g['x1min']} .. {g['x1max']}  "
                                 f"({g['cells_inside']} / {g['cells_outside']} cells clear)"),
        ("P_orb",                f"{m.P_orb:.5e} t_0 = {m.P_orb*m.T_scale/YR:.3f} yr"),
        ("H/r at r_center",      f"{hr:.4f}   = sqrt((gamma-1)(0.5-C'))"),
        ("gas T at mid-plane",   f"{T_mid:.3e} K   <-- NOT T_0; T_0 only sets the units"),
        ("disk mass",           f"{M:.3f} M_sun per L_0 slab  "
                                 f"({M/m.M_bh:.2e} of M_bh)"),
        ("nu at r_center",       f"{m.nu_iso:.4e}"),
        ("tau_visc",             f"{m.tau_visc:.4e} t_0 = {m.tau_visc*m.T_scale/YR:.0f} yr"),
        ("orbits to accrete",    f"{N:.0f}   = 1/(2 pi alpha (H/r)^2)"),
    ]
    w = max(len(k) for k, _ in rows)
    return "\n".join(f"  {k:<{w}}  {v}" for k, v in rows)

print(summarise(m, g))

## Radial structure

In [ ]:
def fig_structure(m, g):
    r = np.linspace(g["x1min"], g["x1max"], 2000)
    fig, ax = plt.subplots(2, 2, figsize=(11.5, 7))

    a = ax[0, 0]
    a.semilogy(r, m.rho(r), color=CAT[0], zorder=3)
    a.axhline(m.rho_atm, color=CAT[1], lw=1.4, zorder=2)
    a.annotate(r"$\rho_{\rm atm}$", (g["x1max"]*0.6, m.rho_atm), color=CAT[1], fontsize=9,
               fontweight="600", xytext=(0, 9), textcoords="offset points", ha="center")
    band(a, m, g); a.set_ylim(m.rho_atm*0.3, 2)
    a.set_ylabel(r"$\rho$  [$\rho_0$]"); a.set_title("(a)  density", loc="left")

    a = ax[0, 1]
    a.semilogy(r, m.p(r), color=CAT[0], zorder=3)
    a.axhline(m.p_atm, color=CAT[1], lw=1.4, zorder=2)
    band(a, m, g)
    a.set_ylabel("$p$  [code]"); a.set_title("(b)  pressure", loc="left")

    a = ax[1, 0]
    vk = np.sqrt(m.beta / r)
    a.plot(r, m.v_phi(r), color=CAT[0], zorder=4)
    a.plot(r, vk, color=CAT[1], ls="--", zorder=3)
    band(a, m, g)
    a.set_ylabel(r"$v_\phi$  [$c_{s0}$]"); a.set_title("(c)  rotation", loc="left")
    a.legend(["equilibrium", "Keplerian"], loc="upper right")

    a = ax[1, 1]
    a.plot(r, r*m.v_phi(r), color=CAT[0], zorder=4)
    a.plot(r, r*vk, color=CAT[1], ls="--", zorder=3)
    band(a, m, g)
    a.set_ylabel(r"$\ell = r v_\phi$"); a.set_title(r"(d)  angular momentum", loc="left")
    a.legend(["equilibrium", "Keplerian"], loc="lower right")

    for a in ax.flat:
        a.set_xlabel("$r$  [$r_0$]")
    fig.tight_layout(); plt.show()

fig_structure(m, g)

### Interactive — disk shape
`gamma` and `C'` are the only two knobs that move `H/r`.

In [ ]:
def explore_shape(gamma=1.3, C_prime=0.2, alpha=1e-3):
    mm = build(gamma=gamma, C_prime=C_prime, alpha=alpha)
    gg = mm.grid(**GRID)
    r = np.linspace(gg["x1min"], gg["x1max"], 1500)
    hr = np.sqrt((gamma - 1)*(0.5 - C_prime))
    N = orbits_to_accrete(alpha, gamma, C_prime)

    fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.6))
    a = ax[0]
    a.semilogy(r, mm.rho(r), color=CAT[0], zorder=3)
    a.axhline(mm.rho_atm, color=CAT[1], lw=1.2, zorder=2)
    band(a, mm, gg); a.set_ylim(mm.rho_atm*0.3, 2)
    a.set_xlabel("$r$  [$r_0$]"); a.set_ylabel(r"$\rho$")
    a.set_title(f"n = {mm.n_poly:.1f},  disk {mm.r_in:.3f}–{mm.r_out:.3f}", loc="left")

    a = ax[1]
    rr = np.linspace(mm.r_in, mm.r_out, 800)
    body = mm.rho_disk(rr) > 10*mm.rho_atm
    a.plot(rr, mm.H_over_r(rr), color=CAT[1], ls="--", zorder=3)
    if body.any():
        a.plot(rr[body], mm.H_over_r(rr[body], disk_only=True), color=CAT[0], zorder=4)
    a.axhline(0.1, color=GUIDE, ls="--", lw=1.0)
    a.plot([mm.r_center], [hr], "o", ms=8, color=CAT[0], mec=SURFACE, mew=2, zorder=5)
    band(a, mm)
    a.set_xlabel("$r$  [$r_0$]"); a.set_ylabel("$H/r$")
    a.set_title(f"H/r = {hr:.3f}   →   {N:.0f} orbits to accrete", loc="left")
    a.legend(["total (ambient past the surface)", "disk only"], loc="upper left")
    fig.tight_layout(); plt.show()

interact(explore_shape,
         gamma=FloatSlider(value=1.3, min=1.02, max=1.7, step=0.02, description="gamma"),
         C_prime=FloatSlider(value=0.2, min=0.05, max=0.49, step=0.01, description="C'"),
         alpha=FloatLogSlider(value=1e-3, base=10, min=-4, max=-1, step=0.1,
                              description="alpha"));

## Force balance
`f_sum` is the finite-difference error of $dp/dr$, not a physical residual.

In [ ]:
def fig_forces(m, g, nx1=None):
    nx1 = nx1 or g["nx1"]
    _, rv = cells(m, g, nx1)
    p_c = m.p(rv)
    f_grav, f_centr = -m.beta/rv**2, m.v_phi(rv)**2/rv
    d = np.empty(nx1)
    d[1:-1] = (p_c[2:] - p_c[:-2]) / (rv[2:] - rv[:-2])
    d[0], d[-1] = (p_c[1]-p_c[0])/(rv[1]-rv[0]), (p_c[-1]-p_c[-2])/(rv[-1]-rv[-2])
    f_press = -d / m.rho(rv)
    f_sum = f_grav + f_centr + f_press
    rel = np.abs(f_sum) / np.abs(f_grav)

    fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.8))
    a = ax[0]
    for y, c in [(f_grav, CAT[0]), (f_centr, CAT[1]), (f_press, CAT[2])]:
        a.plot(rv, y, color=c, zorder=3)
    a.plot(rv, f_sum, color=CAT[3], lw=1.4, zorder=4)
    a.set_yscale("symlog", linthresh=1e2); band(a, m)
    a.set_ylabel("force / mass  [code]")
    a.set_title("(a)  terms", loc="left")
    a.legend([r"$f_{\rm grav}$", r"$f_{\rm centr}$", r"$f_{\rm press}$", r"$f_{\rm sum}$"],
             loc="lower right", ncol=2, columnspacing=1.0)

    a = ax[1]
    a.semilogy(rv, np.maximum(rel, 1e-12), color=CAT[3], zorder=3)
    a.axhline(1e-2, color=GUIDE, ls="--", lw=1.0); band(a, m); a.set_ylim(1e-9, 60)
    a.set_ylabel(r"$|f_{\rm sum}|/|f_{\rm grav}|$")
    a.set_title(f"(b)  residual — peak {rel.max():.1e} at the surface", loc="left")

    for a in ax:
        a.set_xlabel("$r$  [$r_0$]")
    fig.tight_layout(); plt.show()

fig_forces(m, g)

## Grid, mass, timescales, viscosity

In [ ]:
def fig_grid_mass(m, g):
    nx1 = g["nx1"]
    rf, rv = cells(m, g, nx1)
    dr = (g["x1max"] - g["x1min"]) / nx1
    vol = 0.5*(rf[1:]**2 - rf[:-1]**2) * 2*np.pi
    dm = m.rho_disk(rv) * vol * m.mass_scale

    fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.6))
    a = ax[0]
    a.axvspan(m.r_in, m.r_out, color=CAT[0], alpha=0.16, lw=0, zorder=1)
    for x in rf[::max(1, nx1//44)]:
        a.axvline(x, color="#d8d8d4", lw=0.5, zorder=2)
    for x, c, lab, yl, h in [(g["x1min"], CAT[3], "x1min", 0.96, "right"),
                             (m.r_in,     CAT[1], "r_in",  0.60, "left"),
                             (m.r_out,    CAT[1], "r_out", 0.60, "right"),
                             (g["x1max"], CAT[3], "x1max", 0.96, "left")]:
        a.axvline(x, color=c, lw=2.0, zorder=4)
        a.annotate(lab, (x, yl), color=c, fontsize=9, fontweight="600", ha=h, va="center",
                   xytext=(-5 if h == "right" else 5, 0), textcoords="offset points")
    a.set_yticks([]); a.set_ylim(0, 1); a.grid(visible=False)
    a.set_title(f"(a)  grid — dr={dr:.4f}", loc="left")

    a = ax[1]
    a.plot(rv, np.cumsum(dm), color=CAT[0], zorder=3); band(a, m, g)
    a.set_ylabel(r"$M(<r)$  [$M_\odot$]")
    a.set_title(f"(b)  mass — {dm.sum():.2f} $M_\\odot$", loc="left")

    a = ax[2]
    rr = np.linspace(m.r_in, m.r_out, 600)
    nu = m.alpha*m.gamma/np.sqrt(m.beta) * (m.p(rr)/m.rho(rr)) * rr**1.5
    series = [(2*np.pi*np.sqrt(rr**3/m.beta), CAT[0], "orbital"),
              (rr/np.sqrt(m.gamma*m.p(rr)/m.rho(rr)), CAT[1], "sound crossing")]
    if m.alpha > 0:                                  # inviscid runs have no viscous clock
        series.append((rr**2/nu, CAT[2], "viscous"))
    for y, c, _ in series:
        a.semilogy(rr, y, color=c, zorder=3)
    a.set_ylabel("time  [$t_0$]")
    a.set_title("(c)  clocks", loc="left")
    a.legend([s[2] for s in series], loc="center right")

    for a in ax:
        a.set_xlabel("$r$  [$r_0$]")
    fig.tight_layout(); plt.show()

fig_grid_mass(m, g)

### Interactive — run planner
Resolution and viscosity against what the run has to resolve.

In [ ]:
def plan(alpha=1e-3, nx1=176, nx2=128, orbits=100):
    mm = build(alpha=alpha)
    gg = mm.grid(nx1=nx1, nx2=nx2)
    dr = (gg["x1max"] - gg["x1min"]) / nx1
    rdphi = 2*np.pi*mm.r_center/nx2
    hr = np.sqrt((mm.gamma-1)*(0.5-mm.C_prime))
    vK = np.sqrt(mm.beta/mm.r_center)
    dt = 0.4*min(dr, rdphi)/(vK + hr*vK)
    N = orbits_to_accrete(alpha, mm.gamma, mm.C_prime)
    rr = np.linspace(mm.r_in, mm.r_out, 20000)
    hit = mm.rho_disk(rr) > mm.rho_atm
    body = (rr[hit][-1] - rr[hit][0]) if hit.any() else 0.0

    fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.4))
    a = ax[0]
    al = np.logspace(-4, -1, 200)
    a.loglog(al, [orbits_to_accrete(x, mm.gamma, mm.C_prime) for x in al],
             color=CAT[0], zorder=3)
    a.plot([alpha], [N], "o", ms=9, color=CAT[1], mec=SURFACE, mew=2, zorder=5)
    a.axhline(orbits, color=GUIDE, ls="--", lw=1.0)
    a.set_xlabel(r"$\alpha$"); a.set_ylabel("orbits to accrete")
    a.set_title(f"{N:.0f} orbits needed, {orbits} configured "
                f"({orbits/N:.1%})", loc="left")

    # Three checks on one scale: each bar is the value over the level it must clear,
    # so 1.0 is the pass line regardless of which direction "good" points in.
    a = ax[1]
    checks = [("cells across\ndisk body", body/dr, 30.0, "min"),
              ("cells per $H$\nat $r_c$",  hr*mm.r_center/dr, 8.0, "min"),
              ("cell aspect\n$r d\\phi/dr$", rdphi/dr, 3.0, "max")]
    score = [(v/lim if kind == "min" else lim/v) for _, v, lim, kind in checks]
    a.bar(np.arange(3), score, width=0.55,
          color=[CAT[0] if s >= 1 else CAT[3] for s in score], zorder=3)
    a.axhline(1.0, color=INK, lw=1.2, zorder=4)
    for i, ((lab, v, lim, kind), s) in enumerate(zip(checks, score)):
        a.annotate(f"{v:.1f}\n({'≥' if kind == 'min' else '≤'}{lim:g})", (i, s),
                   xytext=(0, 5), textcoords="offset points", ha="center",
                   fontsize=8.5, fontweight="600")
    a.set_xticks(range(3)); a.set_xticklabels([c[0] for c in checks], fontsize=8.5)
    a.set_yscale("log"); a.set_ylim(0.1, max(score)*3); a.grid(axis="x", visible=False)
    a.set_ylabel("value / pass level")
    a.set_title(f"dt ≈ {dt:.2e},  {orbits*mm.P_orb/dt:.2e} steps", loc="left")
    fig.tight_layout(); plt.show()

    print(f"tlim = {orbits*mm.P_orb:.5g}   nu(r_c) = {mm.nu_iso:.4e}   "
          f"cells = {nx1*nx2}")

interact(plan,
         alpha=FloatLogSlider(value=1e-3, base=10, min=-4, max=-1, step=0.1,
                              description="alpha"),
         nx1=IntSlider(value=176, min=64, max=768, step=32, description="nx1"),
         nx2=IntSlider(value=128, min=64, max=768, step=32, description="nx2"),
         orbits=IntSlider(value=100, min=10, max=2000, step=10, description="orbits"));

## Check the generated athinput

In [ ]:
import importlib, verify_athinput
importlib.reload(verify_athinput)

path = "../../inputs/hydro/athinput.acc_disk_visc"
try:
    _, rows = verify_athinput.check(path)
    w = max(len(n) for _, n, _ in rows)
    for s, n, d in rows:
        print(f"[{ {'ok':'  ok  ','warn':' warn ','FAIL':' FAIL '}[s] }] {n:<{w}}  {d}")
except FileNotFoundError:
    print(f"{path} not found — run from scripts/theory/")

---
`python3 disk_model.py` regenerates `inputs/hydro/athinput.acc_disk_visc` and verifies it.
Never hand-edit that file.